# Week 6 — Efficient Inference, Model Compression, and Fine-Tuning

**Course:** High-Performance Computing & Scaling Large Models  
**Practical:** Distributed fine-tuning with LoRA/QLoRA and post-training quantization.

---

## Learning Objectives

1. State the *parameter-efficient fine-tuning* (PEFT) problem and motivate **LoRA** mathematically.
2. Implement a LoRA adapter from scratch and attach it to a `nn.Linear`.
3. Explain **QLoRA**: 4-bit `NF4` quantization, paged optimizers, double quantization.
4. Distinguish *post-training quantization* (PTQ) from *quantization-aware training* (QAT) and survey the dominant PTQ methods (GPTQ, AWQ, SmoothQuant).
5. Configure a distributed QLoRA fine-tune via `peft` + `bitsandbytes` + `accelerate`.
6. Benchmark inference of a quantized model against its full-precision counterpart.


## 1. The Cost of Fine-Tuning

Fine-tuning a 7-billion-parameter model in full precision requires ~112 GB of model state (recall Week 4) plus activations. Even with ZeRO-3 across 8 A100s, you spend roughly $$8 \times \$3 / \text{hr} \times \text{several hours}$$ per experiment. For many downstream tasks — instruction following on a small dataset, domain adaptation, classifier-head replacement — *the model barely changes*. We are updating 7 billion parameters when the *effective rank* of the change is closer to a few hundred.

This is the empirical observation that motivates **Low-Rank Adaptation** (Hu et al., 2021).


## 2. LoRA: Low-Rank Adaptation

Let $W_0 \in \mathbb{R}^{d \times k}$ be a frozen pretrained weight. LoRA represents the update as a low-rank product

$$
W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} \, B A
$$

where $A \in \mathbb{R}^{r \times k}$, $B \in \mathbb{R}^{d \times r}$, and $r \ll \min(d, k)$. Only $A$ and $B$ are trained; $W_0$ is held fixed. The scalar $\alpha / r$ controls the effective learning rate of the adapter.

For $d = k = 4096$ and $r = 16$:

- Full-rank update: $d \cdot k = 16{,}777{,}216$ parameters.
- LoRA: $r(d + k) = 131{,}072$ parameters.
- **Compression ratio: 128×.**

At inference time, $\Delta W$ can be either *merged* into $W_0$ (zero extra latency) or kept *additive* (so multiple adapters can be hot-swapped per request — the trick behind Modal/Hugging Face's adapter routing).

### Why It Works

Empirical studies — first by the LoRA authors, then by many follow-ups — find that the *spectrum* of the fine-tuning update on most language tasks has a sharp knee: the top few hundred singular values dominate. The low-rank parameterization is not a regularizer in the classical sense; it is an *architectural prior* that matches the empirical structure of fine-tuning gradients.


In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")


In [ ]:
# A from-scratch LoRA implementation
class LoRALinear(nn.Module):
    """Wraps a frozen Linear, adding a trainable low-rank update."""
    def __init__(self, base_layer: nn.Linear, r: int = 8, alpha: float = 16.0,
                 dropout: float = 0.0):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False                  # freeze the base

        in_features  = base_layer.in_features
        out_features = base_layer.out_features
        self.r       = r
        self.scaling = alpha / r

        # Standard LoRA initialization: A ~ Kaiming, B = 0  →  ΔW starts at zero
        self.A = nn.Parameter(torch.empty(r, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        # Forward pass: y = W₀x + (α/r) · B A x
        base = self.base_layer(x)
        delta = self.dropout(x) @ self.A.t() @ self.B.t()
        return base + self.scaling * delta

    def merge(self):
        """Folds the LoRA update into the base weight (in-place)."""
        with torch.no_grad():
            self.base_layer.weight += self.scaling * self.B @ self.A
        # Zero out A, B so subsequent forwards don't double-count
        with torch.no_grad():
            self.A.zero_()
            self.B.zero_()


# Smoke test on a 4096 → 4096 linear
torch.manual_seed(0)
base   = nn.Linear(4096, 4096).to(device)
lora   = LoRALinear(base, r=16, alpha=32).to(device)

x      = torch.randn(2, 16, 4096, device=device)
y_base = base(x)
y_lora = lora(x)

print(f"At initialization (B=0): y_lora ≈ y_base ? "
      f"{torch.allclose(y_base, y_lora, atol=1e-5)}")

trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
total     = sum(p.numel() for p in lora.parameters())
print(f"Trainable parameters: {trainable:>10,d}")
print(f"Total parameters    : {total:>10,d}")
print(f"Compression ratio   : {total / trainable:.1f}×")


## 3. QLoRA: 4-bit Backbone, Full-Precision Adapter

QLoRA (Dettmers et al., 2023) takes the next step. Observation: if the *base* weights are frozen, they need not live in FP16 — quantization that would destroy training-time gradients is harmless when no gradients flow through the quantized tensor.

QLoRA's three contributions:

### 3.1 NF4 (Normal Float 4)

Weight tensors of trained networks are *approximately* normally distributed around zero. Standard uniform 4-bit quantization wastes most of its representational budget on values that almost never occur. NF4 places its 16 quantization levels at the *quantiles* of a unit normal, so each level is used with equal frequency. Loss in downstream perplexity vs FP16: typically < 0.1 nats.

### 3.2 Double Quantization

NF4 stores a per-block (typically 64-element) FP32 scale. Those scales themselves are then 8-bit quantized — saving roughly 0.4 bits per parameter at negligible accuracy cost.

### 3.3 Paged Optimizers

Adam's $m$ and $v$ tensors are stored on the CPU and *paged* in and out of GPU memory via NVIDIA Unified Memory. Eliminates the OOM events that occur when a long-sequence batch causes a transient memory spike.

### Headline Result

A **65B-parameter LLaMA** can be QLoRA-fine-tuned on a *single 48 GB GPU*. Without QLoRA, this would require an 8×80 GB node — a 20× cost reduction.


## 4. NF4 Quantization, From Scratch

The cell below implements NF4 in pure PyTorch — enough to grasp the algorithm, not enough to match `bitsandbytes` in speed. For production use, install `bitsandbytes` and let it handle the CUDA kernels.


In [ ]:
# Pure-Python NF4: the 16 quantiles of a unit normal that bitsandbytes uses
NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928, -0.5250730, -0.39491748,
    -0.28444138, -0.18477343, -0.09105003,  0.0,
     0.07958029,  0.16093020,  0.24611230,  0.33791524,
     0.44070983,  0.56261665,  0.72295684,  1.0,
])


def quantize_nf4(w: torch.Tensor, block_size: int = 64):
    """Block-wise NF4 quantization. Returns (codes uint8, scales fp32)."""
    levels = NF4_LEVELS.to(w.device)
    flat   = w.flatten()
    n      = flat.numel()
    pad    = (-n) % block_size
    flat   = F.pad(flat, (0, pad))

    blocks = flat.view(-1, block_size)
    scales = blocks.abs().max(dim=1, keepdim=True).values.clamp_min(1e-8)
    norm   = blocks / scales                          # in [-1, 1]
    # Assign to nearest level
    dists  = (norm.unsqueeze(-1) - levels).abs()
    codes  = dists.argmin(dim=-1).to(torch.uint8)     # in [0, 15]
    return codes, scales.squeeze(1), n, w.shape


def dequantize_nf4(codes, scales, n_orig, shape):
    levels = NF4_LEVELS.to(codes.device)
    blocks = levels[codes.long()] * scales.unsqueeze(1)
    flat   = blocks.flatten()[:n_orig]
    return flat.view(shape)


# Demonstration
W = torch.randn(4096, 4096, device=device)
codes, scales, n, shape = quantize_nf4(W, block_size=64)
W_q                     = dequantize_nf4(codes, scales, n, shape)

err = (W - W_q).abs().mean().item()
nrm = W.abs().mean().item()
print(f"Mean absolute error  : {err:.4e}")
print(f"Mean absolute weight : {nrm:.4e}")
print(f"Relative error       : {err / nrm:.2%}")
print()

# Storage comparison
fp16_bytes = W.numel() * 2
nf4_bytes  = W.numel() / 2 + scales.numel() * 4   # 0.5 byte/param + scales
print(f"FP16 storage : {fp16_bytes / 1e6:.2f} MB")
print(f"NF4  storage : {nf4_bytes / 1e6:.2f} MB")
print(f"Compression  : {fp16_bytes / nf4_bytes:.2f}×")


**Caveats.**

- A 1–2 % relative error is typical and entirely tolerable when the quantized tensor is held *fixed* (as in QLoRA).
- The same error would derail mid-training optimization if propagated through gradients — which is why QLoRA quantizes only the *frozen* base and keeps the LoRA adapters in BF16.
- Block sizes between 32 and 256 are the practical range; 64 is a good default.


## 5. QLoRA in Practice: `peft` + `bitsandbytes`

The production stack:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = "CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
# trainable params: 8,388,608 || all params: 6,750,000,000 || trainable%: 0.12
```

Training proceeds with `Trainer` or `accelerate` as for any other model — except the optimizer step touches only the LoRA parameters, ~0.1 % of the total.

Below is a *runnable miniature* of the same pipeline on a small public model so this notebook completes on modest hardware.


In [ ]:
# Miniature QLoRA pipeline on a tiny model (DistilGPT2). The point is to
# exercise every piece of the stack — the *real* speedup shows on 7B+ models.
import importlib.util

have_peft = importlib.util.find_spec("peft") is not None
have_bnb  = importlib.util.find_spec("bitsandbytes") is not None
have_hf   = importlib.util.find_spec("transformers") is not None

print(f"transformers   : {have_hf}")
print(f"peft           : {have_peft}")
print(f"bitsandbytes   : {have_bnb}")

if have_hf and have_peft:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model, TaskType

    name = "distilgpt2"
    tok  = AutoTokenizer.from_pretrained(name)
    base = AutoModelForCausalLM.from_pretrained(name)

    cfg = LoraConfig(
        r              = 8,
        lora_alpha     = 16,
        target_modules = ["c_attn"],   # GPT-2's combined QKV projection
        lora_dropout   = 0.05,
        bias           = "none",
        task_type      = TaskType.CAUSAL_LM,
    )
    model = get_peft_model(base, cfg)
    model.print_trainable_parameters()
else:
    print("Install `transformers` and `peft` to run this cell.")


## 6. Post-Training Quantization (PTQ) for Inference

LoRA/QLoRA target *training*. For pure *inference*, post-training quantization compresses already-trained weights without any optimizer involvement. The leading techniques:

### GPTQ (Frantar et al., 2022)

Reformulates quantization as layer-wise least-squares: for each weight matrix, find quantized weights $\hat W$ that minimize $\|X \hat W - X W\|_F^2$ on a small calibration set, processing columns greedily with a Cholesky-based update. Targets 3–4 bits with < 1 % perplexity degradation.

### AWQ (Lin et al., 2023) — Activation-Aware Weight Quantization

Observation: only a tiny fraction (~1 %) of weight channels are *salient* — their activations have outlier scale. AWQ identifies these channels and protects them, scaling the rest more aggressively. Faster than GPTQ to apply, comparable quality.

### SmoothQuant (Xiao et al., 2022)

Tackles the harder problem of W8A8 (8-bit weights and *activations*). Outliers in activations are migrated to the weights via a per-channel rescaling, equalizing the dynamic ranges of both tensors.

### Choosing

| Method | Precision | Use case |
|--------|-----------|----------|
| GPTQ   | W4A16    | Highest quality 4-bit; slow to apply |
| AWQ    | W4A16    | Faster application, deployment in TensorRT-LLM and vLLM |
| SmoothQuant | W8A8 | Maximum *throughput* — both tensors fit Tensor Core paths |
| NF4 (bitsandbytes) | W4A16 | Training (QLoRA) and lightweight inference |

Modern serving stacks (vLLM, TensorRT-LLM, SGLang) accept any of these as an input format and dispatch to specialized kernels.


## 7. Inference Benchmark: FP16 vs INT8 vs INT4

We compare three weight-only quantization paths on a small model, measuring tokens/second and memory.


In [ ]:
# Minimal benchmark. Replace `name` with a larger model if your GPU allows.
if have_hf and torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    name = "distilgpt2"
    tok  = AutoTokenizer.from_pretrained(name)
    prompt = "High performance computing enables modern AI by"
    ids = tok(prompt, return_tensors="pt").input_ids.cuda()

    def bench_gen(model, n_iter=5, max_new_tokens=64):
        for _ in range(2):    # warmup
            _ = model.generate(ids, max_new_tokens=max_new_tokens,
                               do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(n_iter):
            out = model.generate(ids, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
        e.record()
        torch.cuda.synchronize()
        ms = s.elapsed_time(e) / n_iter
        return ms, max_new_tokens / (ms / 1000)

    fp16_model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).cuda().eval()
    ms, tps = bench_gen(fp16_model)
    print(f"FP16 : {ms:.1f} ms / 64 tokens  ({tps:.1f} tok/s)")
    print(f"       memory: {torch.cuda.max_memory_allocated()/1e6:.1f} MB")
    del fp16_model; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    if have_bnb:
        # 8-bit weights, FP16 activations
        from transformers import BitsAndBytesConfig
        bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
        int8_model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb_cfg)
        ms, tps = bench_gen(int8_model)
        print(f"\nINT8 : {ms:.1f} ms / 64 tokens  ({tps:.1f} tok/s)")
        print(f"       memory: {torch.cuda.max_memory_allocated()/1e6:.1f} MB")
    else:
        print("\nINT8 / INT4 paths skipped: install bitsandbytes.")
else:
    print("CUDA + transformers required.")


**Trends to expect (on larger models, e.g. 7B–13B):**

- **INT8** ≈ 1.6× higher throughput than FP16, 2× memory reduction, negligible quality loss.
- **INT4 (NF4 / AWQ)** ≈ 2.5–3× throughput, 4× memory reduction, < 1 % quality loss.
- On *small* models like distilgpt2, the overhead of dequantization can *exceed* the savings — quantization pays off in proportion to memory pressure.


## 8. Distributed QLoRA Fine-Tuning

For a 13B fine-tune across 2 GPUs, the full launch looks like:

```bash
accelerate launch \
    --multi_gpu --num_processes=2 \
    scripts/train_qlora.py \
    --base_model meta-llama/Llama-2-13b-hf \
    --dataset alpaca_cleaned \
    --lora_r 16 --lora_alpha 32 \
    --bnb_4bit_quant_type nf4 \
    --bnb_4bit_use_double_quant \
    --bf16_compute \
    --batch_size 4 --grad_accum 8 \
    --epochs 3 --lr 2e-4 \
    --output_dir ./outputs/llama2-13b-alpaca
```

The `accelerate` config (`accelerate config`) selects between DDP, FSDP, and DeepSpeed transparently. With FSDP + QLoRA, the model weights are *both* quantized to 4-bit *and* sharded across GPUs — the two memory savings stack.


## 9. Exercises

**Exercise 6.1 — Effective rank.**  
For a fine-tuned model you have available (any size), compute the singular values of $W - W_0$ for several attention projections. Plot the cumulative energy. Where is the 90 % knee?

**Exercise 6.2 — LoRA target modules.**  
Compare downstream performance when LoRA targets only `q_proj, v_proj` vs all four `q_proj, k_proj, v_proj, o_proj` vs adding the MLP layers. Which gives the best parameter-efficiency tradeoff?

**Exercise 6.3 — NF4 quantization error.**  
Replace `NF4_LEVELS` with a uniform grid in $[-1, 1]$ with 16 points and re-measure quantization error on the same matrix. By how much does NF4 reduce it?

**Exercise 6.4 — Inference benchmark.**  
On a model large enough to make the comparison meaningful (≥ 7B params), measure throughput in FP16, INT8 (bitsandbytes), and INT4 (AWQ via vLLM or AutoAWQ). Report tokens/s and GPU memory.

**Exercise 6.5 — Merging LoRA.**  
Train a small LoRA, evaluate it on a held-out set, then call `merge_and_unload()`. Verify the merged model produces identical outputs to the LoRA-active model on the same input.


## 10. Course Summary

Across six weeks we traced the chain from *hardware* to *algorithms* to *systems*:

1. **Week 1 — Foundations.** The roofline model and the memory wall.
2. **Week 2 — CUDA.** Owning the kernel level when libraries don't suffice.
3. **Week 3 — FlashAttention & vLLM.** Memory-aware attention and KV-cache management.
4. **Week 4 — ZeRO & FSDP.** Sharding model state to break the single-GPU memory ceiling.
5. **Week 5 — 3D Parallelism.** Combining data, tensor, and pipeline parallelism at cluster scale.
6. **Week 6 — Efficient Inference.** LoRA, QLoRA, and post-training quantization for deployment.

The unifying lesson: every order of magnitude in model size has required a corresponding innovation in *systems engineering*, not in mathematics. The next decade's models will be built by teams that understand both halves equally well.

## Further Reading

- Hu, E. J. et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* arXiv:2106.09685.
- Dettmers, T. et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* NeurIPS.
- Frantar, E. et al. (2022). *GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers.* ICLR 2023.
- Lin, J. et al. (2023). *AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration.* MLSys 2024.
- Xiao, G. et al. (2022). *SmoothQuant: Accurate and Efficient Post-Training Quantization for Large Language Models.* ICML 2023.
